# MamiDey AI - Feature Engineering & Synthetic Data
**Input:** `data/processed/ndhs_southwest_clean.csv`

**Output:** `data/processed/X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`  

---
## Architecture
```
FeatureEngineer   -> imputation, encoding, interaction features
SyntheticBuilder  -> generates ANC dropout community population
DatasetAssembler  -> combines NDHS + synthetic, stratified 80/20 split
```

### 1. Import & Config

In [1]:
# Import core  libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import os, json, logging, warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

log = logging.getLogger("MamaDeyAI")

clean_path= "../data/processed/ndhs_southwest_clean_data.csv"
processed_dir = "../data/processed/"
synthetic_dir = "../data/synthetic/"
outputs_dir = "../outputs/"
random_state = 42

os.makedirs(synthetic_dir, exist_ok=True)
log.info("Environment ready")

17:29:00 [INFO] Environment ready


### 2. Feature Engineering

In [2]:
# Feature engineer new data

class FeatureEngineer:
    """ 
    Prepares the NDHS dataset for machine learning
    fit_transform() : fits on NDHS data.
    transform() applies fitted parameters to new data(e.g synthetic).
    """

    drop_cols = [
        "cluster_id", "sample_weight", "delivery_place", "child_alive",
        "residence_label", "education_label", "wealth_label",
        "age_group", "data_source",
    ]

    numeric_median = [
        "anc_visits", "anc_timing_months", "total_children_ever_born",
        "living_children", "age",
    ]

    binary_mode = [
        "anc_any", "anc_4plus", "anc_dropout", "anc_late_entry",
        "anc_provider_doc", "anc_provider_nurse", "anc_provider_tba",
        "anc_blood_pressure", "anc_urine_sample", "anc_iron_tablets",
        "delivery_csection", "tba_any", "skilled_birth",
        "delivery_alone", "postnatal_check",
    ]

    def __init__(self, df: pd.DataFrame):
        # create a copy of the dataset before working on them
        self.df         = df.copy()
        self.impute_values = {}
        self.feature_cols = []


    def col(self, c):
        return c in self.df.columns
    
    def drop(self):
        to_drop = [c for c in self.drop_cols if self.col(c)]
        self.df = self.df.drop(columns=to_drop)
    
    def impute(self, fit):
        # Numeric: median imputation
        for col in self.numeric_median:
            if not self.col:
                continue
            if fit:
                self.impute_values[col] = self.df[col].median()
                self.df[col] = self.df[col].fillna(self.impute_values.get(col, 0))

        # Binary: mode imputation
        for col in self.binary_mode:
            if not self.col:
                continue
            if fit:
                self.impute_values[col] = int(self.df[col].mode()[0])
                self.df[col] = self.df[col].fillna(self.impute_values.get(col, 0))
        
        # Delivery location
        if self.col("delivery_location"):
            self.df["delivery_location"] = self.df["delivery_location"].fillna("other")

        # anc_timing_months: 0 for women with zero ANC visits, median otherwise
        if self.col('anc_timing_months') and self.col('anc_visits'):
            self.df['anc_timing_months'] = self.df.apply(
            lambda row: 0 if row['anc_visits'] == 0
                        else (self.impute_values.get('anc_timing_months', 0)
                              if pd.isna(row['anc_timing_months'])
                              else row['anc_timing_months']),
            axis=1)
        
        # if self.col('anc_timing_months') and self.col('anc_visits'):
        #     self.df['anc_timing_months'] = self.df['anc_timing_months'].fillna(
        #     self.df['anc_visits'].apply(lambda x: 0 if x == 0 else np.nan)
        #     ).fillna(self.impute_values.get('anc_timing_months', 0))

        
        log.info(f"Imputation done - remaining nulls: {self.df.isnull().sum().sum()}")

    def encode(self):
        if self.col("delivery_location"):
            self.df["delivery_location_enc"] = self.df["delivery_location"].map({"public_facility": 0, "private_facility": 1, "other": 2, "home": 3}).fillna(2)
            self.df = self.df.drop(columns=["delivery_location"])
        if self.col("marital_status"):
            self.df["married"] = self.df["marital_status"].isin([1, 2]).astype(int)
            self.df = self.df.drop(columns=["marital_status"])

    def interactions(self):
        if self.col("total_children_ever_born") and self.col("wealth_index"):
            self.df["parity_wealth_risk"] = (self.df["total_children_ever_born"] * (6 - self.df["wealth_index"]))

        if self.col("anc_late_entry") and self.col("anc_dropout"):
            self.df["late_and_dropout"] = ((self.df["anc_late_entry"]== 1) & (self.df["anc_dropout"] ==1)).astype(int)
        
        if self.col("anc_provider_doc") and self.col("skilled_birth"):
            self.df["no_skilled_care"] = (
                (self.df["anc_provider_doc"]== 0) & 
                (self.df.get("anc_provider_nurse", pd.Series(0, index=self.df.index)) == 0) & 
            (self.df["skilled_birth"] ==0)
            ).astype(int)
        if self.col("age"):
            self.df["adolescent"]             = (self.df["age"] < 18).astype(int)
            self.df["advanced_maternal_age"]  = (self.df["age"] > 35).astype(int)
        log.info("Interaction features engineered")
    
    def set_features(self):
        numeric_dtypes = [np.float64, np.int64, np.int32, np.float32, "Int64"]
        self.feature_cols = [
            c for c in self.df.columns 
            if c != "adverse_outcome"
            and str(self.df[c].dtype) in [str(d) for d in numeric_dtypes]
                + ["Int64", "float64", "int32","float32"]                             
        ]
    
    def report(self):
        print("-" * 55)
        print("FEATURE ENGINEERING REPORT")
        print("-" * 55)
        print(f"Shape           :   {self.df.shape}")
        print(f"Features        :   {len(self.feature_cols)}")
        print(f"Remaining nulls :   {self.df.isnull().sum().sum()}")
        if "adverse_outcome" in self.df.columns:
            t = self.df["adverse_outcome"]
            pos = int(t.sum())
            print(f"Adverse cases  : {pos}({pos/len(t)*100:.1f}%)")
            print(f"Imbalance ratio : {int((t==0).sum())/max(pos,1):.1f}:1")
            print(f">> SMOTE to be applied to train set in the next notebook")
            print()
            print("Feature list:")
            for f in self.feature_cols: print(f"    {f}")
            print("-" * 55)



    def fit_transform(self) -> pd.DataFrame:
        self.drop()
        self.impute(fit=True)
        self.encode()
        self.interactions()
        self.set_features()
        self.report()
        return self.df
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        self.df = df.copy()
        self.drop()
        self.impute(fit=False)
        self.encode()
        self.interactions()
        return self.df
        

### 3. SyntheticBuilder

In [3]:
class SyntheticGenerator:
    """ 
    Generates synthetic records for the community-level ANC dropout
    population not captured in the NDHS.

    Features:
    - 0-2 ANC visit (early dropout or none)
    - Predominantly home delivery
    - Higher TBA involvement
    - Skewed toward lower wealth and rural residence
    - Higher adverse outcome rate(calibrated from literature)

    All records labelled data_source="synthetic
    """

    def __init__(self, ndhs_df: pd.DataFrame,
                 no_synthetic: int = 800,
                 random_state: int = 42):
        self.ndhs_df = ndhs_df
        self.no     = no_synthetic
        self.range  = np.random.default_rng(random_state)
        self.df     = None

    def generate(self):
        no      = self.no
        range   = self.range
        df      = self.ndhs_df

        age_mean    = float(df["age"].mean()) if "age" in df.columns else 27
        age_std     = float(df["age"].std()) if "age" in df.columns else 6
        parity_mean = float(df["total_children_ever_born"].mean()) \
                        if "total_children_ever_born" in df.columns else 3
        
        age         = np.clip(range.normal(age_mean, age_std, no), 15, 49).astype(int)
        wealth      = range.choice([1,2,3,4,5], p =[0.35, 0.30, 0.20, 0.10, 0.05], size=no)
        edu         = range.choice([0,1,2,3], p=[0.25, 0.35, 0.30, 0.10], size=no)
        residence   = range.choice([1,2], p=[0.35,0.65], size=no)
        marital     = range.choice([1,2,3,4,5], p=[0.65, 0.10,0.10,0.10,0.05], size=no)
        parity      = np.clip(range.poisson(parity_mean + 0.5, no), 1, 15).astype(int)

        anc_visits  = range.choice([0,1,2], p=[0.40, 0.35, 0.25], size=no)
        anc_timing  = np.where(anc_visits == 0, np.nan, range.choice([4,5,6,7,8], size=no).astype(float))
        anc_any     = (anc_visits > 0).astype(int)
        anc_4plus    = np.zeros(no, dtype=int)
        anc_dropout = ((anc_visits > 0) & (anc_visits < 4)).astype(int)
        anc_late    = ((anc_timing > 3) | np.isnan(anc_timing)).astype(int)
        anc_tba     = range.choice([0,1], p=[0.55, 0.45], size=no)
        anc_doc     = np.zeros(no, dtype=int)
        anc_nurse   = range.choice([0,1], p=[0.80,0.20], size=no)
        anc_bp      = range.choice([0,1], p=[0.60, 0.40], size=no)
        anc_urine   = range.choice([0,1], p=[0.70, 0.30], size=no)
        anc_iron    = range.choice([0,1], p=[0.65,0.35], size=no)

        delivery_location = range.choice(["home", "public_facility", "private_facility", "other"], p=[0.65, 0.20, 0.08, 0.07], size=no)
        delivery_by_csec  = np.zeros(no, dtype=int)
        delivery_nurse    = range.choice([0,1], p=[0.85,0.15], size=no)
        delivery_alone    = range.choice([0,1], p=[0.70, 0.30], size=no)
        tba_any           = range.choice([0,1], p=[0.45, 0.55], size=no)
        skilled           = delivery_nurse.copy()
        postnatal         = range.choice([0,1], p=[0.75,0.25], size=no)

        base_risk  = 0.08
        risk_mod   = (tba_any * 0.04 + (1-skilled) * 0.03 + (anc_visits  ==0).  astype(int) * 0.04)
        adverse_p = np.clip(base_risk + risk_mod, 0, 0.35)
        adverse     = range.binomial(1, adverse_p, no)

        self.df = pd.DataFrame({
            "age":                      age,
            "wealth_index":             wealth,
            "education_level":          edu,
            "residence":                residence,
            "marital_status":           marital,
            "total_children_ever_born": parity,
            "living_children":          np.clip(parity - range.poisson(0.3,no), 0, parity),
            "anc_visits":               anc_visits.astype(float),
            "anc_timing_months":        anc_timing,
            "anc_any":                  anc_any,
            "anc_4plus":                anc_4plus,
            "anc_dropout":              anc_dropout,
            "anc_late_entry":           anc_late,
            "anc_provider_doc":         anc_doc,
            "anc_provider_nurse":       anc_nurse,
            "anc_provider_tba":         anc_tba,
            "anc_blood_pressure":       anc_bp,
            "anc_urine_sample":         anc_urine,
            "anc_iron_tablets":         anc_iron,
            "delivery_location":        delivery_location,
            "delivery_csection":        delivery_by_csec,
            "tba_any":                  tba_any,
            "skilled_birth":            skilled,
            "delivery_alone":           delivery_alone,
            "postnatal_check":          postnatal,
            "adverse_outcome":          adverse,
            "data_source":              "synthetic",
        })

    def validate(self):
        no = len(self.df)
        print("-" * 50)
        print("Synthetic data validation")
        print("-" * 50)
        print(f"Records generated   :   {no:,}")
        print(f"Any ANC             :   {self.df["anc_any"].mean()*100:.1f}%")
        print(f"4+ ANC (except 0%)  :   {self.df["anc_4plus"].mean()*100:.1f}%")
        print(f"Home delivery       :   {(self.df["delivery_location"]=="home").mean()*100:.1f}%")
        print(f"TBA involvement     :   {self.df["tba_any"].mean()*100:.1f}%")
        print(f"Adverse outcome rate:   {self.df["adverse_outcome"].mean()*100:.1f}%")
        print(f"Remaining nulls      :   {self.df.isnull().sum().sum()}")
        print("-" * 50)


    def build(self) -> pd.DataFrame:
        self.generate()
        self.validate()
        return self.df



### 4. DatasetAssembler

In [4]:
class DatasetAssembler:
    """ 
    Combines NDHS with synthetic
    Applies feature engineering to both and produces a stratified 80/20 train/test split. The data source flag is preserved throughout.

    """

    target = "adverse_outcome"

    def __init__(self, ndhs_df, synthetic_df,
                 test_size=0.20, random_state=42):
        self.ndhs_df        = ndhs_df
        self.synthetic_df   = synthetic_df
        self.test_size      = test_size
        self.random_state   = random_state
        self.combined_df    = None
        self.X_train        = self.X_test = self.y_train = self.y_test = None
        self.feature_cols   = None
        self.engineer       = None

    def assemble(self):
        log.info("---Dataset Assembly START---")

        # Fit on NDHS, transform synthetic ith same params
        self.engineer = FeatureEngineer(self.ndhs_df)
        ndhs_feat = self.engineer.fit_transform()
        synthetic_feat = self.engineer.transform(self.synthetic_df)

        # COmmon on common columns
        common = [c for c in ndhs_feat.columns if c in synthetic_feat.columns]
        self.combined_df = pd.concat([ndhs_feat[common], synthetic_feat[common]], ignore_index=True)
        log.info(f"Combined: {len(self.combined_df):,} rows"
                 f"({len(ndhs_feat):,} NDHS + {len(synthetic_feat):,} synthetic)")
        
        # Stratified split
        self.feature_cols = [c for c in self.engineer.feature_cols
                             if c in self.combined_df.columns]
        
        X = self.combined_df[self.feature_cols]
        y = self.combined_df[self.target].astype(int)

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(X,y, test_size=self.test_size, random_state=self.random_state, stratify=y)

        log.info(f"Split: {len(self.X_train):,} train / {len(self.X_test):,} test")
        log.info("---Dataset Assembly Complete")

        return self
    
    def summary(self):
        no = len(self.combined_df)
        print("-" * 60)
        print("Dataset Assembly Summary")
        print("-" * 60)
        print(f"NDHS records        : {len(self.ndhs_df):,}")
        print(f"Synthetic records   : {len(self.synthetic_df):,}")
        print(f"Combined total      : {no:,}")
        print(f"Features            : {len(self.feature_cols)}")
        print()
        print(f"Train set           : {len(self.X_train):,} ({len(self.X_train) / no*100:.0f}%)")
        print(f"Test set            : {len(self.X_test):,} ({len(self.X_test) / no*100:.0f}%)")
        print()
        pos_train = int(self.y_train.sum())
        pos_test    = int(self.y_test.sum())
        print(f"Train adverse       : {pos_train} ({pos_train/len(self.y_train)*100:.1f}%)")
        print(f"Test advers         : {pos_test} ({pos_test/len(self.y_test)*100:.1f}%)")
        print()
        print(" SMOTE will be applied to X_train/y_train in the nextbook")
        print("X_test/y_test are held out and nver modified")
        print("-" * 60)

In [7]:
# ── PATCH: fix anc_timing_months null for zero-ANC women ──────────────────
import types

def _impute_patched(self, fit):
    # Numeric: median imputation
    for col in self.NUMERIC_MEDIAN:
        if not self.col(col): continue
        if fit: self.impute_values[col] = self.df[col].median()
        self.df[col] = self.df[col].fillna(self.impute_values.get(col, 0))

    # Binary: mode imputation
    for col in self.BINARY_MODE:
        if not self.col(col): continue
        if fit: self.impute_values[col] = int(self.df[col].mode()[0])
        self.df[col] = self.df[col].fillna(self.impute_values.get(col, 0))

    # Delivery location
    if self.col('delivery_location'):
        self.df['delivery_location'] = self.df['delivery_location'].fillna('other')

    # anc_timing_months: 0 for women with zero ANC visits, median otherwise
    if self.col('anc_timing_months') and self.col('anc_visits'):
        self.df['anc_timing_months'] = self.df.apply(
            lambda row: 0 if row['anc_visits'] == 0
                        else (self.impute_values.get('anc_timing_months', 0)
                              if pd.isna(row['anc_timing_months'])
                              else row['anc_timing_months']),
            axis=1
        )

    remaining = self.df.isnull().sum().sum()
    log.info(f'Imputation done — remaining nulls: {remaining}')

FeatureEngineer.impute = _impute_patched
log.info('FeatureEngineer._impute patched successfully.')

17:31:10 [INFO] FeatureEngineer._impute patched successfully.


### 5. Run Pipeline

In [8]:
# Load NDHS cleaned data
log.info("Loading the cleaned NDHS dataset")
ndhs_df = pd.read_csv(clean_path)
log.info(f"Loaded:  {ndhs_df.shape[0]:,} rows x {ndhs_df.shape[1]} columns")
ndhs_df.head(3)

17:31:14 [INFO] Loading the cleaned NDHS dataset
17:31:14 [INFO] Loaded:  2,563 rows x 34 columns


,cluster_id,sample_weight,residence,residence_label,age,age_group,education_level,wealth_index,wealth_label,marital_status,total_children_ever_born,living_children,anc_visits,anc_timing_months,anc_any,anc_4plus,anc_dropout,anc_late_entry,anc_provider_doc,anc_provider_nurse,anc_provider_tba,anc_blood_pressure,anc_urine_sample,anc_iron_tablets,delivery_place,delivery_location,delivery_csection,tba_any,skilled_birth,delivery_alone,child_alive,adverse_outcome,postnatal_check,data_source
0,1162,0.47491,1,urban,28,25-34,2,4,richer,1,5,3,16.0,5.0,1,1,0,1,0.0,1.0,0.0,0.0,NaN,1,12,home,0.0,0,0,0,1,0,1.0,NDHS_2018
1,1162,0.47491,1,urban,32,25-34,1,2,poorer,1,4,2,1.0,4.0,1,0,1,1,0.0,0.0,0.0,0.0,0.0,1,22,facility_public,0.0,0,1,0,1,0,1.0,NDHS_2018
2,1162,0.47491,1,urban,27,25-34,2,3,middle,1,3,2,9.0,1.0,1,1,0,0,0.0,0.0,0.0,0.0,4.0,1,22,facility_public,0.0,0,1,0,1,0,0.0,NDHS_2018


In [9]:
# Generate synthetic data
generator = SyntheticGenerator(ndhs_df, no_synthetic=800, random_state=random_state)
synthetic_df = generator.build()

synthetic_path = os.path.join(synthetic_dir, "synthetic_community_records.csv")
synthetic_df.to_csv(synthetic_path, index=False)
log.info(f"Synthetic records saved: {synthetic_path}")

17:31:39 [INFO] Synthetic records saved: ../data/synthetic/synthetic_community_records.csv


--------------------------------------------------
Synthetic data validation
--------------------------------------------------
Records generated   :   800
Any ANC             :   60.6%
4+ ANC (except 0%)  :   0.0%
Home delivery       :   67.0%
TBA involvement     :   55.4%
Adverse outcome rate:   14.4%
Remaining nulls      :   315
--------------------------------------------------


In [ ]:
# Assemble + split
assembler = DatasetAssembler(ndhs_df, synthetic_df, 
                             test_size=0.20, random_state=random_state)
assembler.assemble()
assembler.summary()

In [ ]:
# Save all outputs
assembler.combined_df.to_csv(os.path.join(processed_dir, "modelling_dataset.csv"), index=False)
assembler.X_train.to_csv(os.path.join(processed_dir, "X_train.csv"), index=False)
assembler.X_test.to_csv(os.path.join(processed_dir, "X_test.csv"), index=False)
assembler.y_train.to_csv(os.path.join(processed_dir, "y_train.csv"), index=False, header=True)
assembler.y_test.to_csv(os.path.join(processed_dir, "y_test.csv"), index=False, header=True)

with open(os.path.join(processed_dir, "feature_cols.json"), "w") as f:
    json.dump(assembler.feature_cols, f, indent=2)

log.info("All Outputs saved!")
print()
for filename in ["modelling_dataset.csv", "X_train.csv", "X_test.csv", 
                 "y_train.csv", "y_test.csv", "feature_cols.json"]:
    p = os.path.join(processed_dir, filename)
    print(f"{filename:35s}  {os.path.getsize(p)/1024:.1f} KB")

### 6. Source Comparison Plot

In [ ]:
# NDHS vs Synthetic distribution comparison

fig,axes = plt.subplots(1,3, figsize=(13, 6))
palette = {"NHDS_2018": "#05BA60", "synthetic": "#FA4E10"}

# recover source labels
ndhs_source = ndhs_df[["data_source"]].copy() if "date_source" in ndhs_df.columns \
                else pd.DataFrame({"data_source": ["NDHS_2018"]*len(ndhs_df)})

synthetic_source = synthetic_df[["data_source"]].copy()
source_col      = pd.concat([ndhs_source, synthetic_source], ignore_index=True)

plot_combined = assembler.combined_df.copy()
plot_combined["data_source"] = source_col["data_source"].values

# Plot 1: ANC visits
if "anc_visits" in plot_combined.columns:
    for source, group in plot_combined.groupby("data_source"):
        axes[0].hist(group["anc_visits"].dropna(), bins=range(0,12),
                     alpha=0.6, label=source, color=palette.get(source,"blue"),
                     edgecolor="white")
    axes[0].set_title("ANC visit by data source")
    axes[0].set_xlabel("ANC visits")
    axes[0].legend()

# Plot 2: Adverse outcome rate
if "adverse_outcome" in plot_combined.columns:
    adverse_out = plot_combined.groupby("data_source")["adverse_outcome"].mean()*100
    bars = axes[1].bar(adverse_out.index, adverse_out.values,
                       color=[palette.get(x,"grey") for x in adverse_out.index],
                       edgecolor = "white")
    for bar, value in zip(bars, adverse_out.values):
        axes[1].text(bar.get_x()+bar.get_width()/2, value +0.3,
                     f"{value:.1f}%", ha="center",fontsize=12)
        axes[1].set_title("Adverse outcome rate by source")
        axes[1].set_ylabel("Adverse outcome (%)")

# Plot 3 - Wealth distribution
if "wealth_index" in plot_combined.columns:
    wealth_labels = {1:"poorest", 2:"poorer", 3:"middle", 4:"richer", 5:"richest"}

    for source, group in plot_combined.groupby("data_source"):
        wealth = group["wealth_index"].value_counts(normalize=True).sort_index()*100 
        axes[2].plot(wealth.index.map(wealth_labels), wealth.values, marker="o", label=source, color=palette.get(source, "grey"))
        axes[2].set_title("Wealth distribution by Source")
        axes[2].set_ylabel("% of Sample")
        axes[2].legend()

plt.suptitle("NDHS observed vs synthetic community records", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(outputs_dir, "figure7_ndhs_vs_synthetic_data.png"), bbox_inches="tight", dpi=150)
plt.show()
log.info("Saved figure7_ndhs_vs_synthetic_data.png")


### 7. Checklist

In [ ]:
checks = {
    "NDHS data loaded":                 ndhs_df.shape[0] > 0,
    "Synthetic data generated (800)":   len(synthetic_df) == 800,
    "Feature engineering applied":      assembler.engineer is not None,
    "No nulls in combined dataset":     assembler.combined_df.isnull().sum().sum() == 0,
    "Train/test split completed":        assembler.X_train is not None,
    "Test set is ~20%":                 abs(len(assembler.X_test)/len(assembler.combined_df)-0.20) < 0.01,
    "Target not in feature columns":    "adverse_outcome" not in assembler.X_train.columns,
    "Feature list saved":               os.path.exists(f"{processed_dir}feature_cols.json"),
    "Modelling dataset saved":          os.path.exists(f"{processed_dir}modelling_dataset.csv"),
    "X_train saved":                    os.path.exists(f"{processed_dir}X_train.csv"),
    "Source comparison figure saved":   os.path.exists(f"{outputs_dir}figure7_ndhs_vs_synthetic_data.png"),
}

print("-" * 60)
print("  FEATURE ENGINEERING COMPLETION CHECKLIST")
print("-" * 60)
all_pass = True
for label, status in checks.items():
    icon = "\u2713" if status else "\u2717"
    print(f"  {icon}  {label}")
    if not status: all_pass = False
print("=" * 60)
if all_pass:
    print("  All checks passed. Ready for Model Development.")
    print()
    print("  Files needed for the next notebook:")
    for filename in ["X_train.csv","X_test.csv","y_train.csv","y_test.csv","feature_cols.json"]:
        print(f'    data/processed/{filename}')
else:
    print('  Some checks failed, review output above.')

In [ ]:

null_summary = assembler.combined_df.isnull().sum()
null_cols = null_summary[null_summary > 0].sort_values(ascending=False)

print('Columns with nulls in combined dataset:')
print(null_cols)
print(f'\nTotal null cells: {null_summary.sum()}')
print(f'\nNDHS nulls:      {assembler.combined_df.iloc[:len(ndhs_df)].isnull().sum().sum()}')
print(f'Synthetic nulls: {assembler.combined_df.iloc[len(ndhs_df):].isnull().sum().sum()}')